In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
from rasterio.plot import show

# 0.4 - Merge RegBL and Uranium

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `07_OFEV_uranium`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Uranium concentration raster

In [ ]:
src = rasterio.open(f'{ROOT}data/raw/07_OFEV_uranium/GCBA_Uran.tif')

uranium_data = src.read(1)

print(src.profile)
print(src.bounds)
print(src.crs)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
show(uranium_data, ax=ax, cmap='viridis')
plt.show()


### RegBL

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.1.csv', index_col=0)

df_regbl.head()

## Convert RegBL to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

# Convert to EPSG:3857
df_regbl = df_regbl.to_crs(epsg=3857)

## Overlay uranium raster on RegBL to obtain uranium concentration for each building

In [ ]:
with rasterio.open(f'{ROOT}/data/raw/07_OFEV_uranium/GCBA_Uran.tif') as src:
    uranium_band = src.read(1)
    raster_crs = src.crs
    transform = src.transform

    # Fix FM 04/03/26
    df_regbl = df_regbl[df_regbl.geometry.notna() & ~df_regbl.geometry.is_empty]

    uranium_values = []
    for geom in df_regbl.geometry:
        try:
            row, col = src.index(geom.x, geom.y)
            val = uranium_band[row, col]
        except IndexError:
            val = None  # if point is outside raster bounds
        uranium_values.append(val)

df_regbl["Ura - Concentration uranium"] = uranium_values

In [ ]:
# Convert CRS back to EPSG:2056
df_regbl = df_regbl.to_crs(epsg=2056)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
df_regbl.plot(column='Ura - Concentration uranium', ax=ax, legend=True, cmap='viridis')
plt.show()

In [ ]:
# Convert back to DataFrame
df_regbl = pd.DataFrame(df_regbl.drop(columns='geometry'))

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl, labels=True, fontsize=8)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.2.csv")